In [1]:
import GEOparse

gse2 = GEOparse.get_GEO(
    geo="GSE7670",
    destdir="../data/raw/geo",
)

len(gse2.gsms), list(gse2.gsms.keys())[:5]

30-Jun-2026 22:14:26 DEBUG utils - Directory ../data/raw/geo already exists. Skipping.
30-Jun-2026 22:14:26 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE7nnn/GSE7670/soft/GSE7670_family.soft.gz to ../data/raw/geo/GSE7670_family.soft.gz
100%|██████████| 15.5M/15.5M [00:07<00:00, 2.07MB/s]
30-Jun-2026 22:14:37 DEBUG downloader - Size validation passed
30-Jun-2026 22:14:37 DEBUG downloader - Moving /var/folders/48/j7bwkjs14079sd8gcjdn18cw0000gn/T/tmpcw16i3hi to /Users/adityaanand/dev/cancer-lab/data/raw/geo/GSE7670_family.soft.gz
30-Jun-2026 22:14:37 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE7nnn/GSE7670/soft/GSE7670_family.soft.gz
30-Jun-2026 22:14:37 INFO GEOparse - Parsing ../data/raw/geo/GSE7670_family.soft.gz: 
30-Jun-2026 22:14:37 DEBUG GEOparse - DATABASE: GeoMiame
30-Jun-2026 22:14:37 DEBUG GEOparse - SERIES: GSE7670
30-Jun-2026 22:14:37 DEBUG GEOparse - PLATFORM: GPL96
30-Jun-2026 22:14:37 DEBUG GEOparse - SAMPLE: 

(66, ['GSM185811', 'GSM185812', 'GSM185813', 'GSM185814', 'GSM185815'])

In [3]:
for gsm_name, gsm in list(gse2.gsms.items())[:3]:
    print(gsm_name)
    print(gsm.metadata)
    print()

GSM185811
{'title': ['1N: Adjacent normal part of adenocarcinoma'], 'geo_accession': ['GSM185811'], 'status': ['Public on Aug 30 2007'], 'submission_date': ['Apr 30 2007'], 'last_update_date': ['Aug 18 2014'], 'type': ['RNA'], 'channel_count': ['1'], 'source_name_ch1': ['1N'], 'organism_ch1': ['Homo sapiens'], 'taxid_ch1': ['9606'], 'characteristics_ch1': ['Adjacent normal part of adenocarcinoma', 'sex: female'], 'molecule_ch1': ['total RNA'], 'extract_protocol_ch1': ["RNA preparation and analysis were performed according to the Affymetrix's instructions. The quality of the total RNA for microarray analysis was determined using Spectra Max Plus (Molecular Devices) and had an OD260/OD280 ratio ranging from 1.9 to 2.1."], 'label_ch1': ['biotin'], 'label_protocol_ch1': ['Biotinylated cRNA were prepared according to the standard Affymetrix protocol from 10 ug total RNA (Expression Analysis Technical Manual, 2001, Affymetrix).'], 'hyb_protocol': ['Protocols and reagents for hybridization, w

In [4]:
import pandas as pd
import numpy as np

def extract_gse7670_label(gsm):
    title = gsm.metadata.get("title", [""])[0].lower()
    source = gsm.metadata.get("source_name_ch1", [""])[0].lower()
    characteristics = " ".join(gsm.metadata.get("characteristics_ch1", [])).lower()

    text = " ".join([title, source, characteristics])

    if "adjacent normal" in text or "normal part" in text:
        return "normal"
    elif "tumor part" in text or "tumor" in text:
        return "tumor"
    else:
        return "unknown"

geo2_labels = []

for gsm_id, gsm in gse2.gsms.items():
    geo2_labels.append({
        "gsm_id": gsm_id,
        "title": gsm.metadata.get("title", [""])[0],
        "source_name": gsm.metadata.get("source_name_ch1", [""])[0],
        "sample_type": extract_gse7670_label(gsm),
    })

geo2_labels = pd.DataFrame(geo2_labels)

geo2_labels["sample_type"].value_counts(), geo2_labels.head()

(sample_type
 tumor      29
 normal     28
 unknown     9
 Name: count, dtype: int64,
       gsm_id                                       title source_name  \
 0  GSM185811  1N: Adjacent normal part of adenocarcinoma          1N   
 1  GSM185812            1T: Tumor part of adenocarcinoma          1T   
 2  GSM185813  2N: Adjacent normal part of adenocarcinoma          2N   
 3  GSM185814            2T: Tumor part of adenocarcinoma          2T   
 4  GSM185815  3N: Adjacent normal part of adenocarcinoma          3N   
 
   sample_type  
 0      normal  
 1       tumor  
 2      normal  
 3       tumor  
 4      normal  )

In [5]:
first_gsm = list(gse2.gsms.values())[0]

first_gsm.table.head(), first_gsm.table.columns

(           ID_REF   VALUE ABS_CALL  DETECTION P-VALUE
 0  AFFX-BioB-5_at   496.0        P           0.000662
 1  AFFX-BioB-M_at  1148.5        P           0.000052
 2  AFFX-BioB-3_at   392.5        P           0.000095
 3  AFFX-BioC-5_at  1469.4        P           0.000095
 4  AFFX-BioC-3_at  1115.3        P           0.000081,
 Index(['ID_REF', 'VALUE', 'ABS_CALL', 'DETECTION P-VALUE'], dtype='str'))

In [6]:
expr_cols = []

for gsm_id, gsm in gse2.gsms.items():
    table = gsm.table.copy()

    s = table.set_index("ID_REF")["VALUE"]
    s.name = gsm_id

    expr_cols.append(s)

geo2_probe_expr = pd.concat(expr_cols, axis=1).T

geo2_probe_expr.shape

(66, 22283)

In [9]:
# gsm_id may be a column, the index, or "index" if a prior reset_index() lost the name
if "gsm_id" in geo2_labels.columns:
    geo2_labels = geo2_labels.set_index("gsm_id")
elif "index" in geo2_labels.columns:
    geo2_labels = geo2_labels.set_index("index").rename_axis("gsm_id")

geo2_labels = geo2_labels.loc[geo2_probe_expr.index].rename_axis("gsm_id")

(geo2_probe_expr.index == geo2_labels.index).all(), geo2_labels["sample_type"].value_counts()

(np.True_,
 sample_type
 tumor      29
 normal     28
 unknown     9
 Name: count, dtype: int64)

In [10]:
gse2.gpls.keys()

dict_keys(['GPL96'])

In [11]:
gpl2 = list(gse2.gpls.values())[0]

gpl2.table.head()

,ID,GB_ACC,SPOT_ID,Species Scientific Name,Annotation Date,Sequence Type,Sequence Source,Target Description,Representative Public ID,Gene Title,Gene Symbol,ENTREZ_GENE_ID,RefSeq Transcript ID,Gene Ontology Biological Process,Gene Ontology Cellular Component,Gene Ontology Molecular Function
0,1007_s_at,U48705,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,Affymetrix Proprietary Database,U48705 /FEATURE=mRNA /DEFINITION=HSU48705 Huma...,U48705,discoidin domain receptor tyrosine kinase 1 //...,DDR1 /// MIR4640,780 /// 100616237,NM_001202521 /// NM_001202522 /// NM_001202523...,0001558 // regulation of cell growth // inferr...,0005576 // extracellular region // inferred fr...,0000166 // nucleotide binding // inferred from...
1,1053_at,M87338,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,GenBank,M87338 /FEATURE= /DEFINITION=HUMA1SBU Human re...,M87338,"replication factor C (activator 1) 2, 40kDa",RFC2,5982,NM_001278791 /// NM_001278792 /// NM_001278793...,0000278 // mitotic cell cycle // traceable aut...,0005634 // nucleus // inferred from electronic...,0000166 // nucleotide binding // inferred from...
2,117_at,X51757,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,Affymetrix Proprietary Database,X51757 /FEATURE=cds /DEFINITION=HSP70B Human h...,X51757,heat shock 70kDa protein 6 (HSP70B'),HSPA6,3310,NM_002155,0000902 // cell morphogenesis // inferred from...,0005737 // cytoplasm // inferred from direct a...,0000166 // nucleotide binding // inferred from...
3,121_at,X69699,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,GenBank,X69699 /FEATURE= /DEFINITION=HSPAX8A H.sapiens...,X69699,paired box 8,PAX8,7849,NM_003466 /// NM_013951 /// NM_013952 /// NM_0...,0001655 // urogenital system development // in...,0005634 // nucleus // inferred from direct ass...,0000979 // RNA polymerase II core promoter seq...
4,1255_g_at,L36861,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,Affymetrix Proprietary Database,L36861 /FEATURE=expanded_cds /DEFINITION=HUMGC...,L36861,guanylate cyclase activator 1A (retina),GUCA1A,2978,NM_000409 /// XM_006715073,0007165 // signal transduction // non-traceabl...,0001750 // photoreceptor outer segment // infe...,0005509 // calcium ion binding // inferred fro...


In [12]:
gpl2.table.columns.tolist()

['ID',
 'GB_ACC',
 'SPOT_ID',
 'Species Scientific Name',
 'Annotation Date',
 'Sequence Type',
 'Sequence Source',
 'Target Description',
 'Representative Public ID',
 'Gene Title',
 'Gene Symbol',
 'ENTREZ_GENE_ID',
 'RefSeq Transcript ID',
 'Gene Ontology Biological Process',
 'Gene Ontology Cellular Component',
 'Gene Ontology Molecular Function']